In [ ]:
%pip install vllm chromadb flask-cors flask-socketio

In [1]:
import os
import subprocess

def setup_git_repo(repo_url, base_path="/content"):
    """
    Clones a git repository or pulls updates if it already exists.

    Args:
        repo_url (str): The HTTPS URL of the GitHub repository.
        base_path (str): The local directory where the repo should live.
    """
    # Ensure we are in the correct base directory
    os.chdir(base_path)

    # Extract the repository name from the URL (e.g., 'raise-26')
    repo_dir = repo_url.split("/")[-1].replace(".git", "")
    repo_path = os.path.join(base_path, repo_dir)

    if os.path.exists(repo_path):
        print(f"Directory '{repo_dir}' exists. Pulling latest changes...")
        # Run git pull inside the specific directory
        subprocess.run(["git", "-C", repo_path, "pull"], check=True)
    else:
        print(f"Directory '{repo_dir}' not found. Cloning...")
        subprocess.run(["git", "clone", repo_url], check=True)

    return repo_path

In [4]:
setup_git_repo("https://github.com/pandevim/TriMem.git")

Directory 'TriMem' not found. Cloning...


'/content/TriMem'

In [2]:
%cd /content/TriMem

/content/TriMem


In [9]:
import os
import multiprocessing

# vLLM's EngineCore spawns a subprocess. In Jupyter, the default "fork" start
# method copies the live IPython kernel (with CUDA already initialized) into the
# child, which crashes immediately. Switching to "spawn" starts a clean process.
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

# ------------------------------------------------------------------
# Re-run safety: reset the LLM singleton + free GPU memory so you
# can re-run this notebook without restarting the kernel.
# ------------------------------------------------------------------
try:
    import utils.llm as _llm_mod
    if _llm_mod._backend_instance is not None:
        print("[Setup] Resetting stale LLM singleton …")
        _llm_mod._backend_instance = None
except Exception:
    pass

try:
    import gc, torch
    gc.collect()
    torch.cuda.empty_cache()
    print(f"[Setup] GPU memory freed. "
          f"Available: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")
except Exception:
    pass

print("[Setup] Ready. vLLM will use 'spawn' — no restart needed between runs.")

[Setup] Resetting stale LLM singleton …
[Setup] GPU memory freed. Available: 84.6 GB
[Setup] Ready. vLLM will use 'spawn' — no restart needed between runs.


In [ ]:
from huggingface_hub import login
login()

In [ ]:
%run run_benchmark.py --agent baseline --tasks 1


[INIT] Creating baseline agent …
[LLM] Loading Qwen/Qwen3.5-35B-A3B via vLLM …
[LLM] Initializing vLLM engine (model=Qwen/Qwen3.5-35B-A3B, gpu_mem=0.9, max_len=16384) …
INFO 04-10 00:07:43 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 16384, 'disable_log_stats': True, 'model': 'Qwen/Qwen3.5-35B-A3B'}
INFO 04-10 00:07:44 [model.py:549] Resolved architecture: Qwen3_5MoeForConditionalGeneration
INFO 04-10 00:07:44 [model.py:1678] Using max model len 16384
INFO 04-10 00:07:44 [config.py:281] Setting attention block size to 1056 tokens to ensure that attention page size is >= mamba page size.
INFO 04-10 00:07:44 [config.py:312] Padding mamba page size by 0.76% to ensure that mamba page size and attention page size are exactly equal.
